## <font color='blue'>Projeto 5</font>
## <font color='blue'>LangChain e LLM Open-Source Para Sistema de Perguntas e Respostas</font>

## Instalando e Carregando Pacotes

In [1]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark.
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.9 MB/s eta 0:00:00


In [2]:
!pip install -q accelerate peft bitsandbytes transformers trl datasets langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.4/297.4 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 27.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 14.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 31.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 50.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 kB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 27.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 19.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 84.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 35.6 MB/s eta 0:00:00
     ━━━━

In [3]:
# Imports
import torch
import accelerate
import peft
import bitsandbytes
import transformers
import trl
import datasets
import langchain

In [4]:
# Imports
from trl import SFTTrainer
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import pipeline, TrainingArguments
from peft import AutoPeftModelForCausalLM, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain
import warnings
warnings.filterwarnings('ignore')

## Carregando o Dataset Para o Instruction Fine-Tuning

https://huggingface.co/datasets/nlpie/Llama2-MedTuned-Instructions

In [6]:
# Carrega o dataset
dataset = load_dataset("nlpie/Llama2-MedTuned-Instructions")

Generating train split:   0%|          | 0/200252 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/70066 [00:00<?, ? examples/s]

In [7]:
# Selecionamos as linhas para treino do modelo
dsa_dados_treino = dataset["train"].select(indices = range(1000))

In [8]:
dsa_dados_treino

Dataset({
    features: ['instruction', 'input', 'output', 'source'],
    num_rows: 1000
})

In [9]:
# Selecionamos as linhas para teste do modelo
dsa_dados_teste = dataset["train"].select(indices = range(1000, 1200))

In [10]:
dsa_dados_teste

Dataset({
    features: ['instruction', 'input', 'output', 'source'],
    num_rows: 200
})

## Compreendendo o Formato dos Dados de Texto

In [11]:
# Vamos visualizar 3 pontos de dados
for i in range(3):
    data = dataset['train'][i]
    print(f"Ponto de Dado {i + 1}:")
    print("Instruction:", data['instruction'])
    print("Input:", data['input'])
    print("Output:", data['output'])
    print("\n-----------------------------\n")

Ponto de Dado 1:
Instruction: In your role as a medical professional, address the user's medical questions and concerns.
Input: My relative suffering from secondary lever cancer ( 4th stage as per Allopathic doctor) and primary is in rectum. He is continuously with 103 to 104 degree F fever. Allpathic doctor suggested chemo only after fever subsidises. Is treatment possible at Lavanya & what is the time scale of recover.
Output: Hi, dairy have gone through your question. I can understand your concern. He has rectal cancer with liver metastasis. It is stage 4 cancer. Surgery is not possible at this stage. Only treatment options are chemotherapy and radiotherapy according to type of cancer. Inspite of all treatment prognosis is poor. Life expectancy is not good. Consult your doctor and plan accordingly. Hope I have answered your question, if you have any doubts then contact me at bit.ly/ Chat Doctor. Thanks for using Chat Doctor. Wish you a very good health.

----------------------------

## Automatizando a Criação dos Prompts Para Treinamento do Modelo

In [12]:
# Define a função que recebe um dicionário chamado sample
def dsa_cria_prompt(sample):

    # Define uma string pre_prompt que serve como um modelo para a primeira parte do prompt
    pre_prompt = """[INST]<<SYS>> {instruction}\n"""

    # Concatena o pre_prompt com strings adicionais para formar o prompt completo
    prompt = pre_prompt + "{input}" +"[/INST]"+"\n{output}"

    # Atribui o valor da chave 'instruction' do dicionário sample à variável example_instruction
    example_instruction = sample['instruction']

    # Atribui o valor da chave 'input' do dicionário sample à variável example_input
    example_input = sample['input']

    # Atribui o valor da chave 'output' do dicionário sample à variável example_output
    example_output = sample['output']

    # Cria uma instância de PromptTemplate com o prompt definido anteriormente e as variáveis de entrada
    prompt_template = PromptTemplate(template = prompt,
                                     input_variables = ["instruction", "input", "output"])

    # Utiliza o método format da instância prompt_template para substituir as variáveis
    # no template com os valores específicos
    prompt_unico = prompt_template.format(instruction = example_instruction,
                                          input = example_input,
                                          output = example_output)

    # Retorna o prompt formatado
    return prompt_unico

In [13]:
# Testando a função
prompt = dsa_cria_prompt(dsa_dados_treino[0])
print(prompt)

[INST]<<SYS>> In your role as a medical professional, address the user's medical questions and concerns.
My relative suffering from secondary lever cancer ( 4th stage as per Allopathic doctor) and primary is in rectum. He is continuously with 103 to 104 degree F fever. Allpathic doctor suggested chemo only after fever subsidises. Is treatment possible at Lavanya & what is the time scale of recover.[/INST]
Hi, dairy have gone through your question. I can understand your concern. He has rectal cancer with liver metastasis. It is stage 4 cancer. Surgery is not possible at this stage. Only treatment options are chemotherapy and radiotherapy according to type of cancer. Inspite of all treatment prognosis is poor. Life expectancy is not good. Consult your doctor and plan accordingly. Hope I have answered your question, if you have any doubts then contact me at bit.ly/ Chat Doctor. Thanks for using Chat Doctor. Wish you a very good health.


In [14]:
# Testando a função
prompt = dsa_cria_prompt(dsa_dados_teste[0])
print(prompt)

[INST]<<SYS>> In the clinical text, your objective is to identify relationships between medical problems, treatments, and tests. Medical problems are tagged as @problem$, medical tests as @test$, and treatments as @treatment$. Classify the relationship between two entities as one of the following:
Treatment improves medical problem (TrIP)
Treatment worsens medical problem (TrWP)
Treatment causes medical problem (TrCP)
Treatment is administered for medical problem (TrAP)
Treatment is not administered because of medical problem (TrNAP)
Test reveals medical problem (TeRP)
Test conducted to investigate medical problem (TeCP)
Medical problem indicates medical problem (PIP)
No Relations
Include @treatment$ 50 mgs bid , Aricept 10 mgs qhs , @treatment$ 15 mgs bid , Trazodone 100 mgs qhs .[/INST]
No Relations


## Processo de Quantização

In [15]:
# Ativa o carregamento do modelo base com precisão de 4 bits
use_4bit = True

In [16]:
# Define o dtype para o modelo base
bnb_4bit_compute_dtype = "float16"

In [17]:
# Tipo de quantização
bnb_4bit_quant_type = "nf4"

In [18]:
# Desativa a quantização dupla
use_nested_quant = False

In [19]:
# Define o dtype para computação no PyTorch
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

In [20]:
# Define as configurações
bnb_config = BitsAndBytesConfig(load_in_4bit = use_4bit,
                                bnb_4bit_quant_type = bnb_4bit_quant_type,
                                bnb_4bit_compute_dtype = compute_dtype,
                                bnb_4bit_use_double_quant = use_nested_quant)

In [21]:
# Verifica se a GPU suporta bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("A GPU suporta bfloat16. Acelere o treinamento usando bf16=True")
        print("=" * 80)

A GPU suporta bfloat16. Acelere o treinamento usando bf16=True


## Carregando LLM e Tokenizador

https://huggingface.co/NousResearch/Llama-2-7b-chat-hf

In [22]:
# Nome do LLM
nome_llm = "NousResearch/Llama-2-7b-chat-hf"

In [23]:
# Carrega o tokenizador
tokenizer = AutoTokenizer.from_pretrained("NousResearch/Llama-2-7b-chat-hf")

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [24]:
# Carrega o modelo base com a quantização
modelo = AutoModelForCausalLM.from_pretrained(nome_llm,
                                              quantization_config = bnb_config,
                                              device_map = "auto",
                                              use_cache = False)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

In [25]:
# Usa o token EOS do tokenizador para o pad ao fim de cada sequência
tokenizer.pad_token = tokenizer.eos_token

In [26]:
# Ativa o padding ao final de cada sentença
tokenizer.padding_side = "right"

## Configurando Adaptadores LoRa

In [27]:
# Parâmetros LoRa
dsa_peft_config = LoraConfig(r = 8,
                             lora_alpha = 16,
                             lora_dropout = 0.05,
                             bias = "none",
                             task_type = "CAUSAL_LM")

A quantização representa dados com menos bits, tornando-se uma técnica útil para reduzir o uso de memória e acelerar a inferência, especialmente quando se trata de LLMs.

Depois que um modelo é quantizado, ele normalmente não é treinado DIRETAMENTE para tarefas posteriores porque o treinamento pode ser instável devido à menor precisão dos pesos e ativações. Mas como os métodos PEFT apenas adicionam parâmetros extras treináveis, isso permite treinar um modelo quantizado com um adaptador PEFT na parte superior! Combinar quantização com PEFT pode ser uma boa estratégia para treinar até mesmo os maiores modelos em uma única GPU. Por exemplo, QLoRA é um método que quantiza um modelo em 4 bits e depois o treina com LoRA. Este método permite ajustar um modelo de parâmetros de 65B em uma única GPU de 48GB, por exemplo.

O objetivo do PEFT (Parameter-Efficient Fine-Tuning) é manter a maioria dos parâmetros do modelo pré-treinado fixos e ajustar apenas um pequeno subconjunto de parâmetros para adaptar o modelo a uma tarefa específica.

In [28]:
# Prepara o modelo para treinamento
modelo_dsa = prepare_model_for_kbit_training(modelo)

In [29]:
# Junta o modelo quantizado com os adaptadores LoRa
modelo_dsa = get_peft_model(modelo_dsa, dsa_peft_config)

## Parâmetros do Ajuste Fino

In [30]:
output_model = "modelo_ajustado"

In [31]:
# Argumentos de Treino
dsa_training_arguments = TrainingArguments(output_dir = output_model,
                                           per_device_train_batch_size = 1,
                                           gradient_accumulation_steps = 4,
                                           optim = "paged_adamw_32bit",
                                           learning_rate = 2e-4,
                                           lr_scheduler_type = "cosine",
                                           save_strategy = "epoch",
                                           logging_steps = 10,
                                           num_train_epochs = 3,
                                           max_steps = 150,
                                           fp16 = True)

In [32]:
# Cria o Trainer
# Otimizado para ajustar modelos pré-treinados com conjuntos de dados menores em tarefas
# de aprendizagem supervisionada.
dsa_trainer = SFTTrainer(model = modelo_dsa,
                         peft_config = dsa_peft_config,
                         max_seq_length = 512,
                         tokenizer = tokenizer,
                         packing = True,
                         formatting_func = dsa_cria_prompt,
                         args = dsa_training_arguments,
                         train_dataset = dsa_dados_treino,
                         eval_dataset = dsa_dados_teste)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

## Ajuste Fino do LLM

In [33]:
%%time
dsa_trainer.train()

Step,Training Loss
10,2.496500
20,2.204700
30,1.894000
40,1.842200
50,1.744800
60,1.775200
70,1.641700
80,1.692000
90,1.337200
100,1.470700


CPU times: user 4min 9s, sys: 10.5 s, total: 4min 20s
Wall time: 4min 21s


TrainOutput(global_step=150, training_loss=1.6869860140482584, metrics={'train_runtime': 260.8578, 'train_samples_per_second': 2.3, 'train_steps_per_second': 0.575, 'total_flos': 1.2186386694144e+16, 'train_loss': 1.6869860140482584, 'epoch': 1.05})

In [34]:
# Salva o modelo
dsa_trainer.save_model("modelo_final")

In [35]:
# Merge
merged_model = modelo_dsa.merge_and_unload()

## Construindo o Pipeline de Geração de Texto com LangChain

In [36]:
# Cria o pre-prompt com a instrução
pre_prompt = """[INST] <<SYS>>\nAnalyze the question and answer with the best option.\n"""

In [37]:
# Cria o prompt adicionando o input
prompt = pre_prompt + "Here is my question {context}" + "[\INST]"

In [38]:
# Cria o prompt template com LangChain
prompt = PromptTemplate(template = prompt, input_variables = ["context"])

Os pipelines são uma maneira excelente e fácil de usar modelos para inferência. Esses pipelines são objetos que abstraem a maior parte do código complexo da biblioteca, oferecendo uma API simples dedicada a diversas tarefas, incluindo reconhecimento de entidade nomeada, modelagem de linguagem mascarada, análise de sentimento, extração de recursos e resposta a perguntas.

In [39]:
# Cria o objeto pipeline
dsa_pipe = pipeline("text-generation",
                    model = merged_model,
                    tokenizer = tokenizer,
                    max_new_tokens = 512,
                    use_cache = False,
                    do_sample = True,
                    pad_token_id = tokenizer.eos_token_id,
                    top_p = 0.7,
                    temperature = 0.5)

In [40]:
# Cria o Hugging Face Pipeline
llm_pipeline = HuggingFacePipeline(pipeline = dsa_pipe)

## Criando a LLM Chain

In [41]:
# Cria a memória
memory = ConversationBufferMemory()

In [42]:
# Cria o LLM Chain
dsa_chat_llm_chain = LLMChain(llm = llm_pipeline,
                              prompt = prompt,
                              verbose = False,
                              memory = memory)

## Deploy do Modelo e Uso do Sistema de Perguntas e Respostas

In [43]:
contexto = '''###Question: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:
###Options:
A. Adequate supply of safe drinking water
B. Nutrition
C. Provision of free medicines
D. Basic sanitation'''

In [44]:
%%time
dsa_chat_llm_chain.predict(context = contexto)

CPU times: user 1min 2s, sys: 4.54 s, total: 1min 7s
Wall time: 1min 7s


"[INST] <<SYS>>\nAnalyze the question and answer with the best option.\nHere is my question ###Question: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:\n###Options:\nA. Adequate supply of safe drinking water\nB. Nutrition\nC. Provision of free medicines\nD. Basic sanitation[\\INST]  Great! Let's analyze the question and answer options:\n\nQuestion: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:\n\nOptions:\nA. Adequate supply of safe drinking water\nB. Nutrition\nC. Provision of free medicines\nD. Basic sanitation\n\nThe Alma Ata declaration is a widely recognized international document that outlines the principles of primary health care. According to the declaration, primary health care should include the following provisions:\n\n* Adequate supply of safe drinking water\n* Nutrition\n* Provision of free medicines\n* Basic sanitation\n\nHowever, 

In [46]:
#%watermark -v -m

In [47]:
#%watermark --iversions

# Fim